In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')
print("✅ Imports OK")


✅ Imports OK


In [5]:
DB_PATH = Path("../../02_data_warehouse/churn_dw.duckdb")
conn = duckdb.connect(str(DB_PATH))
df = conn.execute("SELECT * FROM fact_churn").df()
conn.close()

print(f"✅ {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"   Churners : {df['CHURN'].sum():,} ({df['CHURN'].mean()*100:.1f}%)")
print(f"   Actifs   : {(df['CHURN']==0).sum():,} ({(df['CHURN']==0).mean()*100:.1f}%)")

✅ 484,443 lignes × 33 colonnes
   Churners : 221,941 (45.8%)
   Actifs   : 262,502 (54.2%)


In [6]:
FEATURES = [
    'AGE', 'CUST_SENIORITY_YEARS', 'ACCT_BALANCE', 'SALARY',
    'NATURE_CLIENT', 'SCORE_KYC', 'MARITAL_STATUS',
    'CURRENCY', 'NATIONALITY', 'RESIDENCE',
    'LOB', 'INDUSTRY', 'COMPLETED_FILE'
]
TARGET = 'CHURN'

df_ml = df[FEATURES + [TARGET]].copy()
print(f"Dataset ML : {df_ml.shape[0]:,} lignes × {df_ml.shape[1]} colonnes")
print(f"\nValeurs manquantes :")
print(df_ml.isnull().sum()[df_ml.isnull().sum() > 0])

Dataset ML : 484,443 lignes × 14 colonnes

Valeurs manquantes :
AGE                     110770
CUST_SENIORITY_YEARS        81
ACCT_BALANCE            100485
SALARY                  292111
COMPLETED_FILE          207959
dtype: int64


In [7]:
NUM_COLS = ['AGE', 'CUST_SENIORITY_YEARS', 'ACCT_BALANCE', 'SALARY']
CAT_COLS = ['NATURE_CLIENT', 'SCORE_KYC', 'MARITAL_STATUS',
            'CURRENCY', 'NATIONALITY', 'RESIDENCE', 'COMPLETED_FILE']

# Remplir les NaN
for col in NUM_COLS:
    df_ml[col] = df_ml[col].fillna(df_ml[col].median())
for col in CAT_COLS:
    df_ml[col] = df_ml[col].fillna('UNKNOWN')

# Encodage
encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))
    encoders[col] = le

print(f"✅ Nettoyage + encodage terminés")
print(f"   Shape final : {df_ml.shape}")

✅ Nettoyage + encodage terminés
   Shape final : (484443, 14)


In [8]:
X = df_ml.drop(TARGET, axis=1)
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"✅ Train : {X_train.shape[0]:,} | Test : {X_test.shape[0]:,}")
print(f"   Churn train : {y_train.mean()*100:.1f}%")
print(f"   Churn test  : {y_test.mean()*100:.1f}%")

# Sauvegarde
OUT = Path("../../04_machine_learning/models")
OUT.mkdir(parents=True, exist_ok=True)

joblib.dump(scaler,   OUT / "scaler.pkl")
joblib.dump(encoders, OUT / "encoders.pkl")
joblib.dump(
    (X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, FEATURES),
    OUT / "data_splits.pkl"
)
print("✅ Données sauvegardées dans models/")

✅ Train : 387,554 | Test : 96,889
   Churn train : 45.8%
   Churn test  : 45.8%
✅ Données sauvegardées dans models/
